### This notebook converts the simulated waveforms to the frequency domain and normalizes the data for training. Please adjust the number of velocity models as needed. 

In [ ]:
import os
import sys
import torch
import numpy as np
import tqdm

from joblib import dump
sys.path.append("../code")
from utils import process

In [ ]:
nvel = 2  # number of velocity models
nvel_train = 1  # number of velocity models for training, used to build the normalizers
nsrc = 1

In [ ]:
# raw
xrange = 85e3 
zrange = 20e3
h = 125  # spacing in m
nx = int(1.5+xrange/h)
nz = int(1.5+zrange/h)
xcoor = np.arange(nx) * h
zcoor = np.arange(nz) * h

# downsample 
ds = 2
h = 125 * ds  # spacing in m
nx = int(1.5+xrange/h)
nz = int(1.5+zrange/h)
xcoor = np.arange(nx) * h
zcoor = np.arange(nz) * h
assert xcoor[-1] == xrange

# time-freq
start_time_in_seconds = 0  # training actually starts from 0 s
end_time_in_seconds = 50.0
dt = 0.1
i0 = round(-start_time_in_seconds / dt)
fs = 1 / dt
fn = fs / 2  # Nyquist frequency
T = round((end_time_in_seconds - start_time_in_seconds) / dt + 1)
nt = T - i0 
if nt % 2 == 0:
    NT = nt
else:
    NT = nt + 1

freqs = torch.arange(NT // 2 + 1) * fs / (NT - 1)
freqmin = 0.1
freqmax = 0.5
freq_to_keep = torch.where((freqs>=freqmin)&(freqs<=freqmax))[0].tolist()
NF = len(freq_to_keep)

In [ ]:
data_path = "../data_raw/"
save_path_temp = "../data_temp/"
save_path = "../data/"
os.makedirs(save_path_temp, exist_ok=True)
os.makedirs(save_path, exist_ok=True)

### Convert to the frequency domain

In [ ]:
for i in tqdm.trange(nvel):
    vs = np.load(data_path+"vs{}.npy".format(i))[::ds, ::ds]  # (nx, nz)
    vp = np.load(data_path+"vp{}.npy".format(i))[::ds, ::ds]  # (nx, nz)
    vz = np.load(data_path+"vz{}.npy".format(i))  # (nsrc, nrec, nt)
    
    vs = torch.from_numpy(vs).float().flatten()
    vp = torch.from_numpy(vp).float().flatten()
    v = torch.stack([vp, vs], dim=-1)
    np.save(save_path_temp+"v_iv{}.npy".format(i), v)

    one_sample_output = process.convert_to_freq_out(
        torch.from_numpy(vz).float().unsqueeze(-1),
        fs, 
        freqmin, 
        freqmax, 
        taper='cosine'
        )  # (nsrc, nf, nrec, 2)
    for j in range(one_sample_output.shape[0]):  # src
        for k in range(one_sample_output.shape[1]):  # freq
            np.save(save_path_temp+"y_iv{}_js{}_kf{}.npy".format(i, j, k), one_sample_output[j, k])

### Build normalizers

In [ ]:
class OutputNormalizer(object):
    def __init__(self, path_pattern, offset_vel, nvel, nsrc, nfreq):
        # Running stats
        count = 0
        sum_vals = None
        sum_sqs = None

        for i in tqdm.trange(offset_vel, offset_vel + nvel):
            for j in range(nsrc):
                for k in range(nfreq):
                    x = np.load(path_pattern.format(i, j, k))
                    x = torch.from_numpy(x).float()
                    x = x.reshape(*x.shape[:-1], -1, 2)  # (..., nv, 2)
                    x = torch.view_as_complex(x)  # (..., nv)

                    abs_x = x.abs()  # (..., nv)

                    if sum_vals is None:
                        sum_vals = abs_x.sum(dim=tuple(range(abs_x.ndim - 1)))
                        sum_sqs = (abs_x ** 2).sum(dim=tuple(range(abs_x.ndim - 1)))
                        count = abs_x.numel() // abs_x.shape[-1]
                    else:
                        sum_vals += abs_x.sum(dim=tuple(range(abs_x.ndim - 1)))
                        sum_sqs += (abs_x ** 2).sum(dim=tuple(range(abs_x.ndim - 1)))
                        count += abs_x.numel() // abs_x.shape[-1]

        mean = sum_vals / count
        std = (sum_sqs / count - mean ** 2).sqrt()

        nv = mean.shape[0]
        self.mean = torch.zeros(2 * nv)
        self.std = torch.zeros(2 * nv)
        for i in range(nv):
            self.mean[2 * i:2 * i + 2] = mean[i]
            self.std[2 * i:2 * i + 2] = std[i]

    def encode(self, x):
        x.sub_(self.mean).div_(self.std)
        return x

    def decode(self, x):
        x.mul_(self.std).add_(self.mean)
        return x

    def cuda(self):
        self.mean = self.mean.cuda()
        self.std = self.std.cuda()

    def cpu(self):
        self.mean = self.mean.cpu()
        self.std = self.std.cpu()

In [ ]:
y_normalizer = OutputNormalizer(save_path_temp+"y_iv{}_js{}_kf{}.npy", offset_vel=0, nvel=nvel_train, nsrc=nsrc, nfreq=NF)
dump(y_normalizer, save_path+"y_normalizer.sav")

In [ ]:
class InputNormalizer(object):
    def __init__(self, path_pattern, offset, ndp, eps=0):
        # Running stats
        count = 0
        sum_vals = None
        sum_sqs = None

        for i in tqdm.trange(offset, offset + ndp):
            x = np.load(path_pattern.format(i))
            x = torch.from_numpy(x).float()  # (..., nc)

            if sum_vals is None:
                sum_vals = x.sum(dim=tuple(range(x.ndim - 1)))
                sum_sqs = (x ** 2).sum(dim=tuple(range(x.ndim - 1)))
                count = x.numel() // x.shape[-1]
            else:
                sum_vals += x.sum(dim=tuple(range(x.ndim - 1)))
                sum_sqs += (x ** 2).sum(dim=tuple(range(x.ndim - 1)))
                count += x.numel() // x.shape[-1]

        self.mean = sum_vals / count
        self.std = (sum_sqs / count - self.mean ** 2).sqrt()

        self.std[self.std==0] = 1
        self.eps = eps

    def encode(self, x):
        x.sub_(self.mean).div_(self.std + self.eps)  # in-place operations to save memory
        return x

    def decode(self, x):
        x.mul_(self.std + self.eps).add_(self.mean)  # in-place operations to save memory
        return x

    def cuda(self):
        self.mean = self.mean.cuda()
        self.std = self.std.cuda()

    def cpu(self):
        self.mean = self.mean.cpu()
        self.std = self.std.cpu()

In [ ]:
v_normalizer = InputNormalizer(save_path_temp+"v_iv{}.npy", offset=0, ndp=nvel_train)
dump(v_normalizer, save_path+"v_normalizer.sav")

### Normalize data

In [ ]:
for i in tqdm.trange(nvel):
    v = torch.from_numpy(np.load(save_path_temp+"v_iv{}.npy".format(i)))
    v = v_normalizer.encode(v)
    np.save(save_path+"v_iv{}.npy".format(i), v.float())
    os.remove(save_path_temp+"v_iv{}.npy".format(i))

    srcx = np.load(data_path+"srcx{}.npy".format(i))  # (nsrc,)
    srcloc = np.zeros((nsrc, 2))
    srcloc[:, 0] = srcx / xrange
    np.save(save_path+"srcloc_iv{}.npy".format(i), srcloc.astype(np.float32))

    recx = np.load(data_path+"recx{}.npy".format(i))  # (nrec,)
    recloc = np.zeros((len(recx), 2))
    recloc[:, 0] = recx / xrange
    np.save(save_path+"recloc_iv{}.npy".format(i), recloc.astype(np.float32))
    
    for j in range(nsrc):
        for k in range(NF):
            y = torch.from_numpy(np.load(save_path_temp+"y_iv{}_js{}_kf{}.npy".format(i, j, k)))
            y = y_normalizer.encode(y)
            np.save(save_path+"y_iv{}_js{}_kf{}.npy".format(i, j, k), y.float())
            os.remove(save_path_temp+"y_iv{}_js{}_kf{}.npy".format(i, j, k))

os.rmdir(save_path_temp)